In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# OpenQASM 2 și Qiskit SDK

<details>
<summary><b>Versiuni de pachete</b></summary>

Codul de pe această pagină a fost dezvoltat folosind următoarele cerințe.
Recomandăm utilizarea acestor versiuni sau a unora mai noi.

```
qiskit[all]~=2.3.0
```
</details>

Qiskit SDK oferă câteva instrumente pentru a converti între reprezentările OpenQASM ale programelor cuantice și clasa [QuantumCircuit](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit).

<span id="qasm2-import"></span>
## Importă un program OpenQASM 2 în Qiskit
Există două funcții pentru a importa programe OpenQASM 2 în Qiskit.
Acestea sunt [`qasm2.load()`](../api/qiskit/qasm2#load), care primește un nume de fișier, și [`qasm2.loads()`](../api/qiskit/qasm2#loads), care primește programul OpenQASM 2 ca șir de caractere.

In [1]:
import qiskit.qasm2

program = """
    OPENQASM 2.0;
    include "qelib1.inc";
    qreg q[2];
    creg c[2];

    h q[0];
    cx q[0], q[1];

    measure q -> c;
"""
circuit = qiskit.qasm2.loads(program)
circuit.draw()

     ┌───┐     ┌─┐   
q_0: ┤ H ├──■──┤M├───
     └───┘┌─┴─┐└╥┘┌─┐
q_1: ─────┤ X ├─╫─┤M├
          └───┘ ║ └╥┘
c: 2/═══════════╩══╩═
                0  1 

Consultă [API-ul OpenQASM 2 din Qiskit](https://docs.quantum.ibm.com/api/qiskit/qasm2) pentru mai multe informații.

### Importă programe simple
Pentru majoritatea programelor OpenQASM 2, poți folosi `qasm2.load` și `qasm2.loads` cu un singur argument.

#### Exemplu: importă un program OpenQASM 2 ca șir de caractere
Folosește `qasm2.loads()` pentru a importa un program OpenQASM 2 ca șir de caractere într-un QuantumCircuit:

In [2]:
from qiskit import qasm2

program = """
    OPENQASM 2.0;
    include "qelib1.inc";

    qreg q[4];
    creg c[4];

    h q[0];
    cx q[0], q[1];

    // 'rxx' is not actually in `qelib1.inc`,
    // but Qiskit used to behave as if it were.
    rxx(0.75) q[2], q[3];

    measure q -> c;
"""
circuit = qasm2.loads(
    program,
    custom_instructions=qasm2.LEGACY_CUSTOM_INSTRUCTIONS,
)

#### Example: use a particular gate class when importing an OpenQASM 2 program

Qiskit cannot, in general, verify if the definition in an OpenQASM 2 `gate` statement corresponds exactly to a Qiskit standard-library gate.
Instead, Qiskit chooses a custom gate using the precise definition supplied.
This can be less efficient that using one of the built-in standard gates, or a user-defined custom gate.
You can manually define `gate` statements with particular classes.

In [3]:
from qiskit import qasm2
from qiskit.circuit import Gate
from qiskit.circuit.library import RZXGate


# Define a custom gate that takes one qubit and two angles.
class MyGate(Gate):
    def __init__(self, theta, phi):
        super().__init__("my", 1, [theta, phi])


custom_instructions = [
    # Link the OpenQASM 2 name 'my' with our custom gate.
    qasm2.CustomInstruction("my", 2, 1, MyGate),
    # Link the OpenQASM 2 name 'rzx' with Qiskit's
    # built-in RZXGate.
    qasm2.CustomInstruction("rzx", 1, 2, RZXGate),
]

program = """
    OPENQASM 2.0;

    gate my(theta, phi) q {
        U(theta / 2, phi, -theta / 2) q;
    }
    gate rzx(theta) a, b {
        // It doesn't matter what definition is
        // supplied, if the parameters match;
        // Qiskit will still use `RZXGate`.
    }

    qreg q[2];
    my(0.25, 0.125) q[0];
    rzx(pi) q[0], q[1];
"""

circuit = qasm2.loads(
    program,
    custom_instructions=custom_instructions,
)

#### Exemplu: importă un program OpenQASM 2 dintr-un fișier
Folosește `load()` pentru a importa un program OpenQASM 2 dintr-un fișier într-un QuantumCircuit:

In [4]:
from qiskit import qasm2
from qiskit.circuit import Gate


# Define a custom gate that takes one qubit and two angles.
class MyGate(Gate):
    def __init__(self, theta, phi):
        super().__init__("my", 1, [theta, phi])


custom_instructions = [
    qasm2.CustomInstruction("my", 2, 1, MyGate, builtin=True),
]

program = """
    OPENQASM 2.0;
    qreg q[1];

    my(0.25, 0.125) q[0];
"""

circuit = qasm2.loads(
    program,
    custom_instructions=custom_instructions,
)

<span id="custom-instructions"></span>
### Asociază Gate-uri OpenQASM 2 cu Gate-uri Qiskit
În mod implicit, importatorul OpenQASM 2 din Qiskit tratează fișierul include `"qelib1.inc"` ca pe o bibliotecă standard *de facto*.
Importatorul tratează acest fișier ca și cum ar conține exact Gate-urile descrise în [lucrarea originală care definește OpenQASM 2](https://arxiv.org/abs/1707.03429).
Qiskit va folosi Gate-urile integrate din [biblioteca de Circuit-uri](../api/qiskit/circuit_library) pentru a reprezenta Gate-urile din `"qelib1.inc"`.
Gate-urile definite în program prin instrucțiuni manuale `gate` în OpenQASM 2 vor fi, în mod implicit, construite ca subclase personalizate [Qiskit `Gate`](../api/qiskit/qiskit.circuit.Gate).

Poți indica importatorului să folosească anumite clase [`Gate`](../api/qiskit/qiskit.circuit.Gate) pentru instrucțiunile `gate` întâlnite.
De asemenea, poți utiliza acest mecanism pentru a trata nume suplimentare de Gate-uri ca „integrate", adică fără a necesita o definiție explicită.
Dacă specifici ce clase Gate să folosești pentru instrucțiunile `gate` din afara `"qelib1.inc"`, Circuit-ul rezultat va fi în general mai eficient de utilizat.

> **Warning:** Începând cu Qiskit SDK v1.0, *exportatorul* OpenQASM 2 din Qiskit (vezi [Exportă un Circuit Qiskit în OpenQASM 2](#qasm2-export)) se comportă în continuare ca și cum `"qelib1.inc"` ar avea mai multe Gate-uri decât are în realitate.
> Aceasta înseamnă că setările implicite ale importatorului s-ar putea să nu poată importa un program exportat de exportatorul nostru.
> Consultă [exemplul specific de lucru cu exportatorul moștenire](#qasm2-import-legacy) pentru a rezolva această problemă.
> 
> Această discrepanță este un comportament moștenire din Qiskit și [va fi rezolvată într-o versiune ulterioară a Qiskit](https://github.com/Qiskit/qiskit/issues/10737).

Pentru a transmite informații despre o instrucțiune personalizată importatorului OpenQASM 2, folosește [clasa `qasm2.CustomInstruction`](../api/qiskit/qasm2#qiskit.qasm2.CustomInstruction).
Aceasta necesită patru informații obligatorii, în ordine:

* **Numele** Gate-ului, utilizat în programul OpenQASM 2
* **Numărul de parametri unghiulari** pe care Gate-ul îi primește
* **Numărul de Qubiți** pe care Gate-ul acționează
* Clasa sau funcția Python **constructor** pentru Gate, care primește parametrii Gate-ului (dar nu Qubiții) ca argumente individuale

Dacă importatorul întâlnește o definiție `gate` care corespunde unei instrucțiuni personalizate date, va folosi acea informație personalizată pentru a reconstrui obiectul Gate.
Dacă o instrucțiune `gate` corespunde `name`-ului unei instrucțiuni personalizate, dar nu corespunde atât numărului de parametri, cât și numărului de Qubiți, importatorul va genera o eroare [`QASM2ParseError`](../api/qiskit/qasm2#qasm2parseerror), pentru a indica nepotrivirea dintre informațiile furnizate și program.

În plus, un al cincilea argument `builtin` poate fi setat opțional la `True` pentru a face Gate-ul disponibil automat în programul OpenQASM 2, chiar dacă nu este definit explicit.
Dacă importatorul întâlnește o definiție explicită `gate` pentru o instrucțiune personalizată integrată, o va accepta în tăcere.
Ca și înainte, dacă o definiție explicită cu același nume nu este compatibilă cu instrucțiunea personalizată furnizată, va fi generată o eroare [`QASM2ParseError`](../api/qiskit/qasm2#qasm2parseerror).
Aceasta este utilă pentru compatibilitatea cu exportatoarele OpenQASM 2 mai vechi și cu anumite alte platforme cuantice care tratează „Gate-urile de bază" ale hardware-ului lor ca instrucțiuni integrate.

Qiskit furnizează un atribut de date pentru a lucra cu programe OpenQASM 2 produse de versiunile moștenire ale [capabilităților de export OpenQASM 2 din Qiskit](#qasm2-export).
Acesta este [`qasm2.LEGACY_CUSTOM_INSTRUCTIONS`](../api/qiskit/qasm2#legacy-compatibility), care poate fi transmis ca argument `custom_instructions` la [`qasm2.load()`](../api/qiskit/qasm2#load) și [`qasm2.loads()`](../api/qiskit/qasm2#loads).

<span id="qasm2-import-legacy"></span>
#### Exemplu: importă un program creat de exportatorul moștenire din Qiskit
Acest program OpenQASM 2 folosește Gate-uri care nu se află în versiunea originală a `"qelib1.inc"` fără a le declara, dar sunt Gate-uri standard în biblioteca Qiskit.
Poți folosi [`qasm2.LEGACY_CUSTOM_INSTRUCTIONS`](../api/qiskit/qasm2#legacy-compatibility) pentru a indica cu ușurință importatorului să folosească același set de Gate-uri pe care exportatorul OpenQASM 2 din Qiskit l-a utilizat anterior.

In [5]:
import math
from qiskit import qasm2

program = """
    include "qelib1.inc";
    qreg q[2];
    rx(arctan(pi, 3 + add_one(0.2))) q[0];
    cx q[0], q[1];
"""


def add_one(x):
    return x + 1


customs = [
    # Our `add_one` takes only one parameter.
    qasm2.CustomClassical("add_one", 1, add_one),
    # `arctan` takes two parameters, and `math.atan2` implements it.
    qasm2.CustomClassical("arctan", 2, math.atan2),
]
circuit = qasm2.loads(program, custom_classical=customs)

#### Exemplu: folosește o anumită clasă Gate la importul unui program OpenQASM 2
Qiskit nu poate, în general, verifica dacă definiția dintr-o instrucțiune `gate` în OpenQASM 2 corespunde exact unui Gate din biblioteca standard Qiskit.
În schimb, Qiskit alege un Gate personalizat folosind definiția precisă furnizată.
Aceasta poate fi mai puțin eficientă decât utilizarea unuia dintre Gate-urile standard integrate sau a unui Gate personalizat definit de utilizator.
Poți defini manual instrucțiunile `gate` cu anumite clase.

In [6]:
from qiskit import QuantumCircuit, qasm2

# Define any circuit.
circuit = QuantumCircuit(2, 2)
circuit.h(0)
circuit.cx(0, 1)
circuit.measure([0, 1], [0, 1])

# Export to a string.
program = qasm2.dumps(circuit)

# Export to a file.
qasm2.dump(circuit, "my_file.qasm")

#### Exemplu: definește un Gate integrat nou într-un program OpenQASM 2
Dacă argumentul `builtin=True` este setat, un Gate personalizat nu trebuie să aibă o definiție asociată.